In [1]:
!pip install langchain-core
!pip install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 20.5 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.1
    Uninstalling langchain-core-1.6.1:
      Successfully uninstalled langchain-core-1.6.1


In [10]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from google.colab import userdata
import os

In [13]:
# 1. Define the prompt template
query_rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given the chat history and a follow-up question, rephrase the "
         "follow-up as a standalone question that includes all necessary "
         "context from the history. If the question is already standalone, "
         "return it unchanged.\n\n"
         "IMPORTANT: Output ONLY the rephrased question. Do NOT answer it. "
         "Do NOT include any explanation or additional text."),
    # The placeholder waits for a list of messages assigned to 'chat_history'
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

final_answer_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a helpful assistant with a great memory. Answer using the context below and the chat history.
    Use chat history to resolve references like "that and this".
    For factual claims, prioritize the context.
    If information is not available in context or history, say "I don't have that information."

Context:
{context}"""),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}"),
    ])

In [14]:
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_APIKEY')

llm = ChatOpenAI(
        model=os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini'),
        temperature=0,  # Temperature controls randomness (0 = deterministic, 2 = very creative)
        # For RAG, use temperature=0 to ensure consistent, factual responses
        # Reference: https://platform.openai.com/docs/guides/text-generation/how-should-i-set-the-temperature-parameter
        timeout=120,  # Increase timeout for slower connections
        max_retries=3,  # Retry on transient failures
    )

In [22]:
def ask_with_history(question, history):
        if not history:
            # First turn: no history to condense, retrieve directly
            standalone = question
        else:
            # Chain that rewrites the question: dict → standalone question string
            query_rewrite_chain = query_rewrite_prompt | llm | StrOutputParser()

            standalone = query_rewrite_chain.invoke({
                "question": question, "chat_history": history
            })

        # Show what the retriever actually sees (useful for debugging)
        if standalone != question:
            print(f"         [Rewritten query → {standalone}]")


        # Set up the context and generate the answer
        context = "Today is a Sunny Day"

        # Chain that expands the history list into the prompt at call time
        final_answer_chain = final_answer_prompt | llm | StrOutputParser()

        final_answer =  final_answer_chain.invoke({
              "context": context,
              "chat_history": history,
              "question": question,  # Original question, not the rewritten one
            })

        print ("final_answer --> ", final_answer)

        history.append(HumanMessage(content=question))
        history.append(AIMessage(content=final_answer))
        return final_answer



In [26]:
history = []

# Instantiate the parser
strOutputParser = StrOutputParser()

print("\n" + "*"*80)
q1 = "Is it going to rain today ?"
print(f"Turn 1 — User: {q1}")
a1 = ask_with_history(q1, history)
print(f"         Assistant: {a1[:300]}{'...' if len(a1) > 300 else ''}")
print(f"         [chat_history now has {len(history)} messages]")
for message in history:
    if isinstance(message, HumanMessage):
        print(f"         [chat_history HumanMessage now looks like  {message.content} ]")
    elif isinstance(message, AIMessage):
        print(f"         [chat_history AIMessage now looks like  {message.content} ]")
print("\n" + "*"*80)

q2 = "Should i carry umbrella?"
print(f"\nTurn 2 — User: {q2}")
a2 = ask_with_history(q2, history)
print(f"         Assistant: {a2[:300]}{'...' if len(a2) > 300 else ''}")
print(f"         [chat_history now has {len(history)} messages]")
for message in history:
    if isinstance(message, HumanMessage):
        print(f"         [chat_history HumanMessage now looks like  {message.content} ]")
    elif isinstance(message, AIMessage):
        print(f"         [chat_history AIMessage now looks like  {message.content} ]")
print("\n" + "*"*80)

q3 = "Should i wear RainCoat ?"
print(f"\nTurn 3 — User: {q3}")
a3 = ask_with_history(q3, history)
print(f"         Assistant: {a3[:300]}{'...' if len(a3) > 300 else ''}")
print(f"         [chat_history now has {len(history)} messages]")
for message in history:
        print(f"         [chat_history













































































































































































































         now looks like  {message.content} ]")

print("\n" + "*"*80)

print(f"\n✓ History contains {len(history)} messages ({len(history)//2} complete turns)")
print("TIP: In production, cap history to avoid token bloat:")
print("       history = history[-6:]  # keep last 3 turns")


********************************************************************************
Turn 1 — User: Is it going to rain today ?
final_answer -->  No, it's a sunny day today, so it is not going to rain.
         Assistant: No, it's a sunny day today, so it is not going to rain.
         [chat_history now has 2 messages]
         [chat_history HumanMessage now looks like  Is it going to rain today ? ]
         [chat_history AIMessage now looks like  No, it's a sunny day today, so it is not going to rain. ]

********************************************************************************

Turn 2 — User: Should i carry umbrella?
         [Rewritten query → Given that today is a sunny day and it is not going to rain, should I carry an umbrella?]
final_answer -->  No, you don't need to carry an umbrella today since it's a sunny day.
         Assistant: No, you don't need to carry an umbrella today since it's a sunny day.
         [chat_history now has 4 messages]
         [chat_history HumanMes